In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import requests
import datetime
from selenium import webdriver
from time import sleep
import os
from selenium.webdriver.chrome.service import Service as ChromeService
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'KZ AFSAKZ' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running KZ AFSAKZ Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={

        regulatorName + ' 1': 'https://publicreg.myafsa.com/authorised/',
        regulatorName + ' 2': 'https://publicreg.myafsa.com/authorised_mi/',
        regulatorName + ' 3': 'https://publicreg.myafsa.com/ancillary/',
        # Write the logic independently
        regulatorName + ' 4': 'https://publicreg.myafsa.com/rnam/',
        regulatorName + ' 5': 'https://publicreg.myafsa.com/fintech/',
        regulatorName + ' 6': 'https://publicreg.myafsa.com/dasp/',
        }



Typology={

       regulatorName + ' 1': 'Authorised Firms',
       regulatorName + ' 2': 'Authorised Market Institutions',
       regulatorName + ' 3': 'Ancillary Service Providers',
       regulatorName + ' 4': 'Recognised Non-AIFC entities',
       regulatorName + ' 5': 'FinTech Lab Participants',
       regulatorName + ' 6': 'Digital Asset Service Providers',
        }

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36 Edg/143.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,zh;q=0.8,zh-CN;q=0.7",
    "Connection": "keep-alive",
    "Referer": "https://publicreg.myafsa.com/authorised/?company=&license_number=&license_status=active&activities=&products=",
    # If you need the CSRF cookie, include it here:
    # "Cookie": "csrftoken=fjmCRUlAn2pqmDlNDeYsojqpIZ8fOttt",
}

In [5]:
#------------------------------------------------ Function_ ----------------------------------------

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    # if reg == regulatorName+' 1':
    print(reg)
    params = {
        "company": "",
        "license_number": "",
        "license_status": "",
        "activities": "",
        "products": "",
        "page": '',
    }
    resp = requests.get(regdict[reg], params=params, headers=headers, timeout=30,verify=False)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    print("Status:", resp.status_code)
    try:
        pages_elements = soup.find('ul',class_='pagination').find_all('li',class_='page-item')
    except:
        pages_elements = ''
    pages = []
    for page in pages_elements:
        all_page_button = page.find('a').text.strip()
        if all_page_button.isdigit():
            pages.append(all_page_button)

    for page_num in pages:
        update_params = {
        "company": "",
        "license_number": "",
        "license_status": "",
        "activities": "",
        "products": "",
        "page": str(page_num),}
        print(page_num)
        #update_url = regdict[reg] + 'company=&license_number=&license_status=active&activities=&products=&page={page_num}'
        page_resp = requests.get(regdict[reg], params=update_params, headers=headers, timeout=30,verify=False)
        resp.raise_for_status()
        page_soup = BeautifulSoup(page_resp.text, "html.parser")
        table = page_soup.find('table')
        rows = table.select("tbody tr") if table else []
        for row in rows:
            cols = [c.get_text(strip=True) for c in row.find_all("td")]
            if len(cols) >= 6:
                if reg == regulatorName +' 4':
                    company, type_,license_number, issue_date, expired_, status = cols[:6]
                else:
                    company, license_number, status, issue_date, activities, details = cols[:6]
                #print(
                    #     f"Company: {company}\n"
                    #     f"License #: {license_number}\n"
                    #     f"Status: {status}\n"
                    #     f"Issue date: {issue_date}\n"
                    #     f"Activities: {activities}\n"
                    # )
                if 'active' in status.lower():
                    sqldict['Name'].append(company)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['InternalID_1'].append(license_number)
                    sqldict['InternalID_1_type'].append('License Number')
                    sqldict['RegulationDate'].append(issue_date)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)
    else:
        table = soup.find('table')
        rows = table.select("tbody tr") if table else []
        for row in rows:
            cols = [c.get_text(strip=True) for c in row.find_all("td")]
            if len(cols) >= 6:
                company, license_number, status, issue_date, activities, details = cols[:6]
                # print(
                #         f"Company: {company}\n"
                #         f"License #: {license_number}\n"
                #         f"Status: {status}\n"
                #         f"Issue date: {issue_date}\n"
                #         f"Activities: {activities}\n"
                #     )
                if 'active' in status:
                    sqldict['Name'].append(company)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['InternalID_1'].append(license_number)
                    sqldict['InternalID_1_type'].append('License Number')
                    sqldict['RegulationDate'].append(issue_date)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)




[INFO] : Working 1/6 _(KZ AFSAKZ 1)_ 
KZ AFSAKZ 1


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'publicreg.myafsa.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
1


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'publicreg.myafsa.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


2


...

c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'publicreg.myafsa.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
1


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'publicreg.myafsa.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


2


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'publicreg.myafsa.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df = df.drop_duplicates()
df.to_excel(filename, index=False)
driver.quit()
sleep(3)